# qnn-edge-agent: Colab GPU Evaluation Runner

Runs the test suite and evaluates `E0` (FP32), `E1` (`W8A8`), `E2` (naive `W4A8`), `E2a` (`W4A16`), `E2b` (`W16A8`), `E4` (3-boundary SmoothQuant `W4A8`, $\alpha=0.5$), and `E4s` (3-boundary SmoothQuant `W4A8`, searched $\alpha^*$), followed by per-scenario (`pc` vs `iot` vs `bfcl`) diagnostics.

In [1]:
# 1. GPU Check
!nvidia-smi

Wed Oct 28 05:44:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             45W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# 2. Clone repository
import os
GIT_URL = "https://github.com/iamqazolp/qnn-edge-agent.git"
REPO_NAME = "qnn-edge-agent"

if not os.path.exists(REPO_NAME):
    !git clone $GIT_URL $REPO_NAME
%cd $REPO_NAME
!git pull

/content/qnn-edge-agent
Already up to date.


In [3]:
!git checkout -- .
!git pull origin feat/mobile-edge-console
!pip install -e . --no-deps

From https://github.com/iamqazolp/qnn-edge-agent
 * branch            main       -> FETCH_HEAD
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint: 
hint:   git config pull.rebase false  # merge (the default strategy)
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint: 
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.
Obtaining file:///content/qnn-edge-agent
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies

In [4]:
# 3. Install dependencies in editable mode
!pip install -e ".[dev]"

Obtaining file:///content/qnn-edge-agent
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for qnn-edge-agent (pyproject.toml) ... done
  Created wheel for qnn-edge-agent: filename=qnn_edge_agent-0.1.0-py3-none-any.whl size=1601 sha256=09ebd01e111f9df951382639c9a6e2fef480f0e4ed8dd4ecbe03d2d1c71c384a
  Stored in directory: /tmp/pip-ephem-wheel-cache-aspvqb5_/wheels/a5/3c/01/a2308bcb67905f82c340d79da3f01a9e19ecea81b9f0a330d6
Successfully built qnn-edge-agent
  Attempting uninstall: qnn-edge-agent
    Found existing installation: qnn-edge-agent 0.1.0
    Uninstalling qnn-edge-agent-0.1.0:
      Successfully uninstalled qnn-edge-agent-0.1.0


In [5]:
# 4. Run verification tests & inspect recipes
!pytest tests/
!qea recipes

.................................................................        [100%]
65 passed in 5.81s
ID     | Weights    | Acts     | Smooth   | Alpha    | Status
--------------------------------------------------------------
E0     | FP32       | FP32     | none     | -        | ready
E1     | W8         | A8       | none     | -        | ready
E2     | W4         | A8       | none     | -        | ready
E2a    | W4         | A16      | none     | -        | ready
E2b    | FP32       | A8       | none     | -        | ready
E4     | W4         | A8       | ABC      | 0.5      | ready
E4s    | W4         | A8       | ABC      | search   | ready


In [6]:
from pathlib import Path

p = Path("data/eval_tool_calls.jsonl")
fixed = p.read_text().replace(
    '"data": {"description": "The training data for the model."}',
    '"data": {"type": "string", "description": "The training data for the model."}',
)
p.write_text(fixed)

584257

In [7]:
# 5. Run ablation suite across E0, E1, E2, E2a, E2b, E4, E4s
!python scripts/run_ablation_suite.py --ids E0,E1,E2,E2a,E2b,E4,E4s --batch-size 32 --modes unconstrained,constrained

Skipping E0 (already exists at /content/qnn-edge-agent/results/E0.json)
Skipping E1 (already exists at /content/qnn-edge-agent/results/E1.json)
Skipping E2 (already exists at /content/qnn-edge-agent/results/E2.json)
Skipping E2a (already exists at /content/qnn-edge-agent/results/E2a.json)
Skipping E2b (already exists at /content/qnn-edge-agent/results/E2b.json)
Skipping E4 (already exists at /content/qnn-edge-agent/results/E4.json)
Skipping E4s (already exists at /content/qnn-edge-agent/results/E4s.json)
All requested recipe runs exist. Extracting failure examples if available...
Extracted 5 W4A8 failure diffs to /content/qnn-edge-agent/results/w4a8_failure_examples.json


In [8]:
# 6. Summary comparison table + per-scenario (PC vs IoT vs BFCL) diagnostics
from pathlib import Path
import json

print(f"{'Recipe':<8} | {'Avg Bits':<8} | {'Footprint':<10} | {'PPL':<8} | {'Mode':<14} | {'JSON Valid':<11} | {'Exact Val':<11}")
print("-" * 80)
for rid in ["E0", "E1", "E2", "E2a", "E2b", "E4", "E4s"]:
    p = Path(f"results/{rid}.json")
    if not p.exists():
        continue
    with open(p) as f:
        d = json.load(f)
    for mode, sc in d.get("eval", {}).items():
        ppl_str = f"{d.get('wikitext2_ppl', 0.0):.2f}" if d.get('wikitext2_ppl') else "N/A"
        print(f"{rid:<8} | {d['avg_bits_linear']:<8.1f} | {d['footprint_mb']:<7.1f} MB | {ppl_str:<8} | {mode:<14} | {sc['json_valid']['rate']*100:<10.1f}% | {sc['values_exact']['rate']*100:<10.1f}%")

print("\n--- Scenario Breakdown (PC vs IoT vs BFCL) ---")
!qea diagnose --results-dir results

Recipe   | Avg Bits | Footprint  | PPL      | Mode           | JSON Valid  | Exact Val  
--------------------------------------------------------------------------------
E0       | 32.0     | 988.0   MB | 16.07    | unconstrained  | 99.1      % | 70.9      %
E1       | 8.0      | 494.1   MB | 433.81   | unconstrained  | 0.0       % | 0.0       %
E2       | 4.0      | 315.1   MB | 4347.52  | unconstrained  | 0.0       % | 0.0       %
E2a      | 4.0      | 315.1   MB | 37.71    | unconstrained  | 0.9       % | 0.0       %
E2b      | 32.0     | 988.0   MB | 423.88   | unconstrained  | 0.0       % | 0.0       %
E4       | 4.0      | 315.1   MB | 87.65    | unconstrained  | 0.0       % | 0.0       %
E4s      | 4.0      | 315.1   MB | 70.87    | unconstrained  | 0.0       % | 0.0       %

--- Scenario Breakdown (PC vs IoT vs BFCL) ---
Recipe  | Mode          | Overall  | PC       | IoT      | BFCL-S   | BFCL-M  
--------------------------------------------------------------------------
E0   